# ASR Pipeline: Video to Text with Timestamps

Uses **faster-whisper** to transcribe ultrasound teaching videos.

Pipeline: Video (.mp4) → ffmpeg → Audio (.wav) → Whisper → JSON

从字幕里面提取：
- meta data
- 当前超声的描述：出现了什么，说明了什么
- 下一步想要什么操作
- 下一步需要看什么组织

1. 视频的切分
2. Clip：是否匹配，是否是prior knowledge，Q&A生成

In [ ]:
import sys
sys.path.append('../scripts')
from asr_pipeline import *
import json
from pathlib import Path

In [ ]:
VIDEO_PATH = "../UltrasoundCrawler_KeyCode_20260323_v2/output/20260520_162816_youtube/media/case_reasoning/8V649L5Q368.mp4"
assert Path(VIDEO_PATH).exists()
print(f"Video: {VIDEO_PATH}")

## Step 1: Extract Audio

In [ ]:
audio_path = extract_audio(VIDEO_PATH, "transcripts/audio/8V649L5Q368.wav")
print(f"Audio saved: {audio_path}")

## Step 2: Transcribe

In [ ]:
result = transcribe_audio(audio_path, model_size="base")
print(f"Language: {result['language']} | Segments: {len(result['segments'])} | Speed: {result['speed_factor']}x")

In [ ]:
# View segments
for seg in result['segments'][:20]:
    print(f"[{seg['start']:6.1f} - {seg['end']:6.1f}] {seg['text']}")

In [ ]:
# Full text
full_text = ' '.join(s['text'] for s in result['segments'])
print(f"Total length: {len(full_text)} chars")
print(f"\nFirst 500 chars:\n{full_text[:500]}")

## Step 3: Full Pipeline (one call)

In [ ]:
# Or use the full pipeline function
output = transcribe_video(VIDEO_PATH, model_size="base", output_dir="transcripts")

In [ ]:
# Check saved file
saved = json.load(open("transcripts/8V649L5Q368.json"))
print(f"Saved transcript: {saved['num_segments']} segments, {saved['duration_sec']}s")
print(f"Language: {saved['language']}")
print(f"\nFirst 3 segments:")
for s in saved['segments'][:3]:
    print(f"  [{s['start']}-{s['end']}s] {s['text']}")